# T33 - 信用债规模

## 第3章：数据采集与ETL

---

### 数据采集流程

本章节实现债券数据的采集和ETL处理，包括：
1. 获取待处理日期列表
2. 插入市场信息数据（余额、久期）
3. 计算并补全缺失久期
4. 更新隐含评级
5. 执行规模统计汇总

In [ ]:
# 导入依赖
import pandas as pd
import sqlalchemy
from sqlalchemy import text, create_engine
from datetime import datetime
import sqlalchemy.pool as pool
import os

# 从环境变量获取配置
MYSQL_HOST = os.environ.get('MYSQL_HOST', 'rm-uf6c8yi6zdq6ea7p1qo.mysql.rds.aliyuncs.com')
MYSQL_PORT = os.environ.get('MYSQL_PORT', '3306')
MYSQL_USER = os.environ.get('MYSQL_USER', 'hz_work')
MYSQL_PASSWORD = os.environ.get('MYSQL_PASSWORD', '')
MYSQL_DATABASE = os.environ.get('MYSQL_DATABASE', 'bond')

# 创建数据库连接
connection_string = f'mysql+pymysql://{MYSQL_USER}:{MYSQL_PASSWORD}@{MYSQL_HOST}:{MYSQL_PORT}/{MYSQL_DATABASE}'
sql_engine = create_engine(connection_string, poolclass=pool.NullPool)

def log_progress(message):
    current_time = datetime.now().strftime("%Y-%m-%d %H:%M:%S")
    print(f"[{current_time}] {message}")

log_progress("数据库连接创建成功")

### Step 1: 获取待处理日期列表

获取marketinfo_abs中存在但marketinfo3中不存在的日期列表

In [ ]:
def get_unprocessed_dates():
    """获取待处理的日期列表"""
    sql = """
    SELECT DISTINCT dt FROM bond.marketinfo_abs
    WHERE dt NOT IN (
        SELECT DISTINCT dt FROM bond.marketinfo3 
        WHERE ths_bond_balance_bond IS NOT NULL
    )
    """
    with sql_engine.connect() as conn:
        result = pd.read_sql(sql, conn)
    return result['dt'].tolist()

dt_list = get_unprocessed_dates()
log_progress(f"待处理日期数量: {len(dt_list)}")
if dt_list:
    log_progress(f"日期范围: {min(dt_list)} ~ {max(dt_list)}")

### Step 2: 插入信用债市场数据

插入信用债的余额和久期数据到marketinfo3表

In [ ]:
def insert_credit_market_data(dt_list):
    """插入信用债市场数据"""
    if not dt_list:
        log_progress("没有新的信用债数据需要处理")
        return
    
    dt_tuple = tuple(dt_list) if len(dt_list) > 1 else f"('{dt_list[0]}')"
    
    sql = f"""
    INSERT INTO bond.marketinfo3 (DT, trade_code, ths_bond_balance_bond, 久期)
    SELECT 
        DT, 
        A.trade_code, 
        ths_bond_balance_bond,
        CASE 
            WHEN ths_evaluate_modified_dur_cb_bond_exercise > 0 
            THEN ths_evaluate_modified_dur_cb_bond_exercise
            ELSE ths_evaluate_interest_durcb_bond_exercise + ths_evaluate_interest_durcb_bond_exercise 
        END AS 久期
    FROM bond.marketinfo_credit A
    INNER JOIN basicinfo_all B ON A.trade_code = B.trade_code
    WHERE ths_bond_balance_bond > 0
    AND A.dt IN {dt_tuple}
    ON DUPLICATE KEY UPDATE 
        ths_bond_balance_bond = VALUES(ths_bond_balance_bond),
        久期 = VALUES(久期)
    """
    
    with sql_engine.connect() as conn:
        conn.execute(text(sql))
        conn.commit()
    
    log_progress("完成信用债市场数据插入")

insert_credit_market_data(dt_list)

In [ ]:
def update_credit_market_data():
    """更新信用债缺失的余额和久期数据"""
    sql = """
    INSERT INTO bond.marketinfo3 (DT, trade_code, ths_bond_balance_bond, 久期)
    SELECT 
        A.DT, 
        A.trade_code, 
        A.ths_bond_balance_bond,
        CASE 
            WHEN ths_evaluate_modified_dur_cb_bond_exercise > 0 
            THEN ths_evaluate_modified_dur_cb_bond_exercise
            ELSE ths_evaluate_interest_durcb_bond_exercise + ths_evaluate_interest_durcb_bond_exercise 
        END AS 久期
    FROM bond.marketinfo_credit A
    INNER JOIN basicinfo_all B ON A.trade_code = B.trade_code
    INNER JOIN marketinfo3 C ON A.trade_code = C.trade_code AND A.dt = C.dt
    WHERE A.ths_bond_balance_bond > 0
    AND (C.ths_bond_balance_bond IS NULL OR C.久期 IS NULL)
    ON DUPLICATE KEY UPDATE 
        ths_bond_balance_bond = VALUES(ths_bond_balance_bond),
        久期 = VALUES(久期)
    """
    
    with sql_engine.connect() as conn:
        conn.execute(text(sql))
        conn.commit()
    
    log_progress("完成信用债数据更新")

update_credit_market_data()

### Step 3: 补全信用债久期

对于久期缺失的记录，从基础信息表计算久期

In [ ]:
def calculate_credit_duration():
    """计算并补全信用债久期"""
    # 删除临时表
    sql_drop = "DROP TEMPORARY TABLE IF EXISTS temp_table"
    with sql_engine.connect() as conn:
        conn.execute(text(sql_drop))
        conn.commit()
    
    log_progress("开始计算信用债久期...")
    
    # 创建临时表
    sql_create = """
    CREATE TEMPORARY TABLE temp_table AS
    SELECT 
        A.dt,
        B.trade_code,
        CASE
            WHEN B.ths_bond_maturity_theory_bond LIKE '%(%)%' 
                AND SUBSTRING_INDEX(SUBSTRING_INDEX(B.ths_bond_maturity_theory_bond, '(', -1), '+', 1) REGEXP '^[0-9]+$' 
                AND DATEDIFF(DATE_ADD(B.ths_interest_begin_date_bond, 
                    INTERVAL CAST(SUBSTRING_INDEX(SUBSTRING_INDEX(B.ths_bond_maturity_theory_bond, '(', -1), '+', 1) AS UNSIGNED) YEAR), A.DT) / 365 >= 0 
            THEN
                DATEDIFF(DATE_ADD(B.ths_interest_begin_date_bond, 
                    INTERVAL CAST(SUBSTRING_INDEX(SUBSTRING_INDEX(B.ths_bond_maturity_theory_bond, '(', -1), '+', 1) AS UNSIGNED) YEAR), A.DT) / 365
            ELSE
                DATEDIFF(B.ths_maturity_date_bond, A.DT) / 365
        END AS new_duration
    FROM marketinfo3 A
    JOIN basicinfo_credit B ON A.trade_code = B.trade_code
    WHERE A.久期 IS NULL
    """
    
    # 更新久期
    sql_update = """
    UPDATE marketinfo3 A
    JOIN temp_table T ON A.trade_code = T.trade_code AND A.dt = T.dt
    SET A.久期 = T.new_duration
    WHERE A.久期 IS NULL
    """
    
    with sql_engine.connect() as conn:
        conn.execute(text(sql_create))
        conn.execute(text(sql_update))
        conn.commit()
    
    log_progress("完成信用债久期补全")

calculate_credit_duration()

### Step 4: 插入金融债市场数据

In [ ]:
def insert_finance_market_data(dt_list):
    """插入金融债市场数据"""
    if not dt_list:
        log_progress("没有新的金融债数据需要处理")
        return
    
    dt_tuple = tuple(dt_list) if len(dt_list) > 1 else f"('{dt_list[0]}')"
    
    sql = f"""
    INSERT INTO bond.marketinfo3 (DT, trade_code, ths_bond_balance_bond, 久期)
    SELECT 
        DT, 
        A.trade_code, 
        ths_bond_balance_bond,
        CASE 
            WHEN ths_evaluate_modified_dur_cb_bond_exercise > 0 
            THEN ths_evaluate_modified_dur_cb_bond_exercise
            ELSE ths_evaluate_interest_durcb_bond_exercise + ths_evaluate_interest_durcb_bond_exercise 
        END AS 久期
    FROM bond.marketinfo_finance A
    INNER JOIN basicinfo_all B ON A.trade_code = B.trade_code
    WHERE ths_bond_balance_bond > 0
    AND A.dt IN {dt_tuple}
    ON DUPLICATE KEY UPDATE 
        ths_bond_balance_bond = VALUES(ths_bond_balance_bond),
        久期 = VALUES(久期)
    """
    
    with sql_engine.connect() as conn:
        conn.execute(text(sql))
        conn.commit()
    
    log_progress("完成金融债市场数据插入")

insert_finance_market_data(dt_list)

In [ ]:
def update_finance_market_data():
    """更新金融债缺失的余额和久期数据"""
    sql = """
    INSERT INTO bond.marketinfo3 (DT, trade_code, ths_bond_balance_bond, 久期)
    SELECT 
        A.DT, 
        A.trade_code, 
        A.ths_bond_balance_bond,
        CASE 
            WHEN ths_evaluate_modified_dur_cb_bond_exercise > 0 
            THEN ths_evaluate_modified_dur_cb_bond_exercise
            ELSE ths_evaluate_interest_durcb_bond_exercise + ths_evaluate_interest_durcb_bond_exercise 
        END AS 久期
    FROM bond.marketinfo_finance A
    INNER JOIN basicinfo_all B ON A.trade_code = B.trade_code
    INNER JOIN marketinfo3 C ON A.trade_code = C.trade_code AND A.dt = C.dt
    WHERE A.ths_bond_balance_bond > 0
    AND (C.ths_bond_balance_bond IS NULL OR C.久期 IS NULL)
    ON DUPLICATE KEY UPDATE 
        ths_bond_balance_bond = VALUES(ths_bond_balance_bond),
        久期 = VALUES(久期)
    """
    
    with sql_engine.connect() as conn:
        conn.execute(text(sql))
        conn.commit()
    
    log_progress("完成金融债数据更新")

update_finance_market_data()

In [ ]:
def calculate_finance_duration():
    """计算并补全金融债久期"""
    sql_drop = "DROP TEMPORARY TABLE IF EXISTS temp_table"
    with sql_engine.connect() as conn:
        conn.execute(text(sql_drop))
        conn.commit()
    
    log_progress("开始计算金融债久期...")
    
    sql_create = """
    CREATE TEMPORARY TABLE temp_table AS
    SELECT 
        A.dt,
        B.trade_code,
        CASE
            WHEN B.ths_bond_maturity_theory_bond LIKE '%(%)%' 
                AND SUBSTRING_INDEX(SUBSTRING_INDEX(B.ths_bond_maturity_theory_bond, '(', -1), '+', 1) REGEXP '^[0-9]+$' 
                AND DATEDIFF(DATE_ADD(B.ths_interest_begin_date_bond, 
                    INTERVAL CAST(SUBSTRING_INDEX(SUBSTRING_INDEX(B.ths_bond_maturity_theory_bond, '(', -1), '+', 1) AS UNSIGNED) YEAR), A.DT) / 365 >= 0 
            THEN
                DATEDIFF(DATE_ADD(B.ths_interest_begin_date_bond, 
                    INTERVAL CAST(SUBSTRING_INDEX(SUBSTRING_INDEX(B.ths_bond_maturity_theory_bond, '(', -1), '+', 1) AS UNSIGNED) YEAR), A.DT) / 365
            ELSE
                DATEDIFF(B.ths_maturity_date_bond, A.DT) / 365
        END AS new_duration
    FROM marketinfo3 A
    JOIN basicinfo_finance B ON A.trade_code = B.trade_code
    WHERE A.久期 IS NULL
    """
    
    sql_update = """
    UPDATE marketinfo3 A
    JOIN temp_table T ON A.trade_code = T.trade_code AND A.dt = T.dt
    SET A.久期 = T.new_duration
    WHERE A.久期 IS NULL
    """
    
    with sql_engine.connect() as conn:
        conn.execute(text(sql_create))
        conn.execute(text(sql_update))
        conn.commit()
    
    log_progress("完成金融债久期补全")

calculate_finance_duration()

### Step 5: 插入ABS市场数据

In [ ]:
def insert_abs_market_data(dt_list):
    """插入ABS市场数据"""
    if not dt_list:
        log_progress("没有新的ABS数据需要处理")
        return
    
    dt_tuple = tuple(dt_list) if len(dt_list) > 1 else f"('{dt_list[0]}')"
    
    sql = f"""
    INSERT INTO bond.marketinfo3 (DT, trade_code, ths_bond_balance_bond, 久期)
    SELECT 
        DT, 
        A.trade_code, 
        ths_bond_balance_bond,
        CASE 
            WHEN ths_evaluate_modified_dur_cb_bond_exercise > 0 
            THEN ths_evaluate_modified_dur_cb_bond_exercise
            ELSE ths_evaluate_interest_durcb_bond_exercise + ths_evaluate_interest_durcb_bond_exercise 
        END AS 久期
    FROM bond.marketinfo_abs A
    INNER JOIN basicinfo_all B ON A.trade_code = B.trade_code
    WHERE ths_bond_balance_bond > 0
    AND A.dt IN {dt_tuple}
    ON DUPLICATE KEY UPDATE 
        ths_bond_balance_bond = VALUES(ths_bond_balance_bond),
        久期 = VALUES(久期)
    """
    
    with sql_engine.connect() as conn:
        conn.execute(text(sql))
        conn.commit()
    
    log_progress("完成ABS市场数据插入")

insert_abs_market_data(dt_list)

In [ ]:
def update_abs_market_data():
    """更新ABS缺失的余额和久期数据"""
    sql = """
    INSERT INTO bond.marketinfo3 (DT, trade_code, ths_bond_balance_bond, 久期)
    SELECT 
        A.DT, 
        A.trade_code, 
        A.ths_bond_balance_bond,
        CASE 
            WHEN ths_evaluate_modified_dur_cb_bond_exercise > 0 
            THEN ths_evaluate_modified_dur_cb_bond_exercise
            ELSE ths_evaluate_interest_durcb_bond_exercise + ths_evaluate_interest_durcb_bond_exercise 
        END AS 久期
    FROM bond.marketinfo_abs A
    INNER JOIN basicinfo_all B ON A.trade_code = B.trade_code
    INNER JOIN marketinfo3 C ON A.trade_code = C.trade_code AND A.dt = C.dt
    WHERE A.ths_bond_balance_bond > 0
    AND (C.ths_bond_balance_bond IS NULL OR C.久期 IS NULL)
    ON DUPLICATE KEY UPDATE 
        ths_bond_balance_bond = VALUES(ths_bond_balance_bond),
        久期 = VALUES(久期)
    """
    
    with sql_engine.connect() as conn:
        conn.execute(text(sql))
        conn.commit()
    
    log_progress("完成ABS数据更新")

update_abs_market_data()

In [ ]:
def calculate_abs_duration():
    """计算并补全ABS久期"""
    sql_drop = "DROP TEMPORARY TABLE IF EXISTS temp_table"
    with sql_engine.connect() as conn:
        conn.execute(text(sql_drop))
        conn.commit()
    
    log_progress("开始计算ABS久期...")
    
    sql_create = """
    CREATE TEMPORARY TABLE temp_table AS
    SELECT 
        A.dt,
        B.trade_code,
        CASE
            WHEN B.ths_bond_maturity_theory_bond LIKE '%(%)%' 
                AND SUBSTRING_INDEX(SUBSTRING_INDEX(B.ths_bond_maturity_theory_bond, '(', -1), '+', 1) REGEXP '^[0-9]+$' 
                AND DATEDIFF(DATE_ADD(B.ths_interest_begin_date_bond, 
                    INTERVAL CAST(SUBSTRING_INDEX(SUBSTRING_INDEX(B.ths_bond_maturity_theory_bond, '(', -1), '+', 1) AS UNSIGNED) YEAR), A.DT) / 365 >= 0 
            THEN
                DATEDIFF(DATE_ADD(B.ths_interest_begin_date_bond, 
                    INTERVAL CAST(SUBSTRING_INDEX(SUBSTRING_INDEX(B.ths_bond_maturity_theory_bond, '(', -1), '+', 1) AS UNSIGNED) YEAR), A.DT) / 365
            ELSE
                DATEDIFF(B.ths_maturity_date_bond, A.DT) / 365
        END AS new_duration
    FROM marketinfo3 A
    JOIN basicinfo_abs B ON A.trade_code = B.trade_code
    WHERE A.久期 IS NULL
    """
    
    sql_update = """
    UPDATE marketinfo3 A
    JOIN temp_table T ON A.trade_code = T.trade_code AND A.dt = T.dt
    SET A.久期 = T.new_duration
    WHERE A.久期 IS NULL
    """
    
    with sql_engine.connect() as conn:
        conn.execute(text(sql_create))
        conn.execute(text(sql_update))
        conn.commit()
    
    log_progress("完成ABS久期补全")

calculate_abs_duration()

### Step 6: 更新隐含评级

In [ ]:
def get_unprocessed_rating_dates():
    """获取需要更新隐含评级的日期列表"""
    sql = """
    SELECT DISTINCT dt FROM bond.marketinfo_abs
    WHERE dt NOT IN (
        SELECT DISTINCT dt FROM bond.marketinfo3 
        WHERE 隐含评级 IS NOT NULL
    )
    """
    with sql_engine.connect() as conn:
        result = pd.read_sql(sql, conn)
    return result['dt'].tolist()

rating_dt_list = get_unprocessed_rating_dates()
log_progress(f"需要更新隐含评级的日期数量: {len(rating_dt_list)}")

In [ ]:
def update_implicit_rating(dt_list):
    """更新隐含评级数据"""
    if not dt_list:
        log_progress("没有新的隐含评级数据需要处理")
        return
    
    dt_tuple = tuple(dt_list) if len(dt_list) > 1 else f"('{dt_list[0]}')"
    
    sql = f"""
    INSERT INTO bond.marketinfo3 (dt, trade_code, 隐含评级) 
    SELECT A.dt, A.trade_code, A.ths_cb_market_implicit_rating_bond AS 隐含评级 
    FROM bond.marketinfo_credit A
    INNER JOIN basicinfo_all B ON A.trade_code = B.trade_code
    WHERE A.ths_cb_market_implicit_rating_bond IS NOT NULL
    AND A.ths_cb_market_implicit_rating_bond != ''
    AND A.dt IN {dt_tuple}
    
    UNION
    
    SELECT A.dt, A.trade_code, A.ths_cb_market_implicit_rating_bond AS 隐含评级 
    FROM bond.marketinfo_finance A
    INNER JOIN basicinfo_all B ON A.trade_code = B.trade_code
    WHERE A.ths_cb_market_implicit_rating_bond IS NOT NULL
    AND A.ths_cb_market_implicit_rating_bond != ''
    AND A.dt IN {dt_tuple}
    
    UNION
    
    SELECT A.dt, A.trade_code, A.ths_cb_market_implicit_rating_bond AS 隐含评级 
    FROM bond.marketinfo_abs A
    INNER JOIN basicinfo_all B ON A.trade_code = B.trade_code
    WHERE A.ths_cb_market_implicit_rating_bond IS NOT NULL
    AND A.ths_cb_market_implicit_rating_bond != ''
    AND A.dt IN {dt_tuple}
    
    ON DUPLICATE KEY UPDATE 隐含评级 = VALUES(隐含评级)
    """
    
    with sql_engine.connect() as conn:
        conn.execute(text(sql))
        conn.commit()
    
    log_progress("完成隐含评级数据更新")

update_implicit_rating(rating_dt_list)

### ETL处理完成验证

In [ ]:
# 验证marketinfo3表数据
sql_check = """
SELECT 
    COUNT(*) as total_records,
    COUNT(DISTINCT dt) as distinct_dates,
    COUNT(DISTINCT trade_code) as distinct_bonds,
    SUM(CASE WHEN 久期 IS NOT NULL THEN 1 ELSE 0 END) as has_duration,
    SUM(CASE WHEN 隐含评级 IS NOT NULL THEN 1 ELSE 0 END) as has_rating
FROM bond.marketinfo3
WHERE dt >= DATE_SUB(CURDATE(), INTERVAL 30 DAY)
    """

with sql_engine.connect() as conn:
    result = pd.read_sql(sql_check, conn)

print("marketinfo3表最近30天数据统计:")
print(result.to_string())

---

### ETL处理流程完成

**已完成的处理步骤**:
1. 获取待处理日期列表
2. 插入信用债市场数据（余额、久期）
3. 补全信用债久期
4. 插入金融债市场数据
5. 补全金融债久期
6. 插入ABS市场数据
7. 补全ABS久期
8. 更新隐含评级

---

**下一章**: [4_数据处理与特征工程](4_数据处理与特征工程.ipynb)